In [1]:
pip install pandas numpy scikit-learn sentence-transformers gradio

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("harshitshankhdhar/imdb-dataset-of-top-1000-movies-and-tv-shows")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-of-top-1000-movies-and-tv-shows' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-of-top-1000-movies-and-tv-shows


In [5]:
import pandas as pd
df = pd.read_csv(path + "/imdb_top_1000.csv")
df.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


In [7]:
import re
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["Overview"] = df["Overview"].apply(clean_text)


In [8]:
titles = df["Series_Title"].tolist()
genres = df["Genre"].tolist()
overviews = df["Overview"].tolist()
ratings = df["IMDB_Rating"].tolist()

In [10]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Loading Transformer Model...")

model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

Loading Transformer Model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [11]:
print("Creating Movie Embeddings...")

movie_embeddings = model.encode(
    overviews,
    show_progress_bar=True
)

print("Embeddings Ready!")

Creating Movie Embeddings...


Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Embeddings Ready!


In [31]:
all_genres = set()

for genres in df["Genre"]:

    for genre in genres.split(","):

        all_genres.add(genre.strip())

genre_choices = ["All"] + sorted(list(all_genres))

In [32]:
import numpy as np
def movie_chat(query, genre, history):

    filtered_df = df.copy()

    # genre filter
    if genre != "All":

        filtered_df = filtered_df[
            filtered_df["Genre"].str.contains(
                genre,
                case=False,
                na=False
            )
        ]

    if len(filtered_df) == 0:

        response = "No movies found."

        history.append((query, response))

        return history, history

    # filtered embeddings
    filtered_embeddings = model.encode(
        filtered_df["Overview"].tolist()
    )

    # encode query
    query_embedding = model.encode([query])

    # similarity
    similarities = cosine_similarity(
        query_embedding,
        filtered_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:5]

    response = ""

    filtered_df = filtered_df.reset_index(drop=True)

    for idx in top_indices:

        movie = filtered_df.iloc[idx]

        response += f"""
🎬 {movie['Series_Title']}

🎭 Genre: {movie['Genre']}

⭐ Rating: {movie['IMDB_Rating']}

📝 {movie['Overview']}

🔥 Similarity: {similarities[idx]:.4f}

-----------------------------------

"""

    history.append((query, response))

    return history, history

In [33]:
with gr.Blocks() as demo:

    gr.Markdown(
        "# 🎬 Transformer Movie Chatbot"
    )

    chatbot = gr.Chatbot(height=500)

    state = gr.State([])

    with gr.Row():

        query = gr.Textbox(
            placeholder="movies about mafia",
            label="Movie Query"
        )

        genre = gr.Dropdown(
            choices=genre_choices,
            value="All",
            label="Genre Filter"
        )

    submit_btn = gr.Button("Search")

    submit_btn.click(
        fn=movie_chat,

        inputs=[
            query,
            genre,
            state
        ],

        outputs=[
            chatbot,
            state
        ]
    )

# =====================================================
# RUN APP
# =====================================================

demo.launch()

/tmp/ipykernel_1424/3874700639.py:7: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=500)
/tmp/ipykernel_1424/3874700639.py:7: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=500)


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0fe998f9eb24cf6138.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [34]:
np.save(
    "movie_embeddings.npy",
    movie_embeddings
)

print("Embedding saved!")

Embedding saved!


In [35]:
from google.colab import files

files.download("movie_embeddings.npy")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>